In [7]:
#train.csv로 모델을 학습, 학습한 모델을 test.csv에 적용
#bank marketing 참여 여부(정기 예금 가입 여부)를 예측하는 classification tast.

In [ ]:
#특성 엔지니어링 (변수 손질)

In [1]:
import warnings
warnings.filterwarnings('ignore')

In [2]:
!pip install catboost

,Requirement already satisfied: matplotlib in c:\users\gram\anaconda3\lib\site-packages (from catboost) (3.8.3)
,Requirement already satisfied: pandas>=0.24 in c:\users\gram\anaconda3\lib\site-packages (from catboost) (2.2.1)
,Requirement already satisfied: numpy<2.0,>=1.16.0 in c:\users\gram\anaconda3\lib\site-packages (from catboost) (1.26.4)
,Requirement already satisfied: plotly in c:\users\gram\anaconda3\lib\site-packages (from catboost) (5.24.1)
,Requirement already satisfied: six in c:\users\gram\anaconda3\lib\site-packages (from catboost) (1.16.0)
,Requirement already satisfied: graphviz in c:\users\gram\anaconda3\lib\site-packages (from catboost) (0.20.3)
,Requirement already satisfied: scipy in c:\users\gram\anaconda3\lib\site-packages (from catboost) (1.13.0)
,Requirement already satisfied: tzdata>=2022.7 in c:\users\gram\anaconda3\lib\site-packages (from pandas>=0.24->catboost) (2024.1)
,Requirement already satisfied: pytz>=2020.1 in c:\users\gram\anaconda3\lib\site-package

In [28]:
import numpy as np
import os
import pandas as pd
from catboost import CatBoostClassifier

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer

from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import f1_score
from sklearn.metrics import confusion_matrix
from sklearn.metrics import roc_auc_score
from sklearn.metrics import ConfusionMatrixDisplay
from sklearn.metrics import RocCurveDisplay

from sklearn.model_selection import train_test_split
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import RandomizedSearchCV

from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import BaggingClassifier
import lightgbm as lgbm
from catboost import CatBoostClassifier
from sklearn.ensemble import BaggingClassifier, VotingClassifier

In [29]:
df_trn = pd.read_csv('train.csv')
df_tst=pd.read_csv('test.csv')

In [30]:
df_trn.head()

,ID,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,label
0,train00001,34,blue-collar,married,primary,no,358,yes,no,unknown,23,may,100,4,-1,0,unknown,0
1,train00002,33,blue-collar,married,secondary,no,-53,yes,no,unknown,20,may,172,1,-1,0,unknown,0
2,train00003,32,management,single,tertiary,no,207,yes,no,cellular,2,feb,56,1,-1,0,unknown,0
3,train00004,37,blue-collar,divorced,secondary,no,638,yes,no,cellular,8,may,326,1,326,2,success,0
4,train00005,33,housemaid,married,secondary,no,826,yes,no,cellular,11,may,256,1,-1,0,unknown,0


In [31]:
df_trn.info()

<class 'pandas.core.frame.DataFrame'>
,RangeIndex: 31647 entries, 0 to 31646
,Data columns (total 18 columns):
, #   Column     Non-Null Count  Dtype 
,---  ------     --------------  ----- 
, 0   ID         31647 non-null  object
, 1   age        31647 non-null  int64 
, 2   job        31647 non-null  object
, 3   marital    31647 non-null  object
, 4   education  31647 non-null  object
, 5   default    31647 non-null  object
, 6   balance    31647 non-null  int64 
, 7   housing    31647 non-null  object
, 8   loan       31647 non-null  object
, 9   contact    31647 non-null  object
, 10  day        31647 non-null  int64 
, 11  month      31647 non-null  object
, 12  duration   31647 non-null  int64 
, 13  campaign   31647 non-null  int64 
, 14  pdays      31647 non-null  int64 
, 15  previous   31647 non-null  int64 
, 16  poutcome   31647 non-null  object
, 17  label      31647 non-null  int64 
,dtypes: int64(8), object(10)
,memory usage: 4.3+ MB


In [32]:
month_map = {
    'jan': 1, 'feb': 2, 'mar': 3, 'apr': 4, 'may': 5, 'jun': 6,
    'jul': 7, 'aug': 8, 'sep': 9, 'oct': 10, 'nov': 11, 'dec': 12
}

df_trn['month'] = df_trn['month'].map(month_map)
df_tst['month'] = df_tst['month'].map(month_map)

In [33]:
df_trn.head()

,ID,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,label
0,train00001,34,blue-collar,married,primary,no,358,yes,no,unknown,23,5,100,4,-1,0,unknown,0
1,train00002,33,blue-collar,married,secondary,no,-53,yes,no,unknown,20,5,172,1,-1,0,unknown,0
2,train00003,32,management,single,tertiary,no,207,yes,no,cellular,2,2,56,1,-1,0,unknown,0
3,train00004,37,blue-collar,divorced,secondary,no,638,yes,no,cellular,8,5,326,1,326,2,success,0
4,train00005,33,housemaid,married,secondary,no,826,yes,no,cellular,11,5,256,1,-1,0,unknown,0


In [34]:
df_trn['balance'].describe()

count     31647.000000
mean       1370.050084
std        3122.054996
min       -4057.000000
25%          73.000000
50%         451.000000
75%        1418.500000
max      102127.000000
Name: balance, dtype: float64

In [35]:
df_trn['age'].describe()

count    31647.000000
mean        40.916833
std         10.621773
min         18.000000
25%         33.000000
50%         39.000000
75%         48.000000
max         95.000000
Name: age, dtype: float64

In [36]:

print(df_trn['job'].unique())

['blue-collar' 'management' 'housemaid' 'admin.' 'technician' 'services'
, 'retired' 'student' 'unknown' 'unemployed' 'entrepreneur' 'self-employed']


In [37]:
y_cols=['label']
num_cols=['age', 'balance', 'day', 'month', 'duration', 'campaign', 'pdays', 'previous']
cat_cols=['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'poutcome']
id_cols=['ID']
x_cols=num_cols+cat_cols

In [38]:
train_corr=df_trn[num_cols].corr()
high_corr=train_corr[train_corr.abs()>=0.3]
print(high_corr)
#0.4에서 0.6이면 보통의 상관관계라 애매.

          age  balance  day  month  duration  campaign     pdays  previous
,age       1.0      NaN  NaN    NaN       NaN       NaN       NaN       NaN
,balance   NaN      1.0  NaN    NaN       NaN       NaN       NaN       NaN
,day       NaN      NaN  1.0    NaN       NaN       NaN       NaN       NaN
,month     NaN      NaN  NaN    1.0       NaN       NaN       NaN       NaN
,duration  NaN      NaN  NaN    NaN       1.0       NaN       NaN       NaN
,campaign  NaN      NaN  NaN    NaN       NaN       1.0       NaN       NaN
,pdays     NaN      NaN  NaN    NaN       NaN       NaN  1.000000  0.432882
,previous  NaN      NaN  NaN    NaN       NaN       NaN  0.432882  1.000000


In [39]:
df_trn_x = df_trn[x_cols]
df_tst_x = df_tst[x_cols]

df_trn_y = df_trn[y_cols]
df_tst_y = df_tst[y_cols]

In [41]:
import warnings
import pandas as pd

# Suppress the SettingWithCopyWarning
warnings.simplefilter('ignore', category=pd.errors.SettingWithCopyWarning)


In [42]:
# 1. 연령대(age) 범주화
df_trn_x['age_group'] = pd.cut(df_trn_x['age'], bins=[0, 30, 60, 100], labels=['young', 'middle_aged', 'senior'])
df_tst_x['age_group'] = pd.cut(df_tst_x['age'], bins=[0, 30, 60, 100], labels=['young', 'middle_aged', 'senior'])

# 2. 잔고(balance) 범주화
df_trn_x['balance_group'] = pd.cut(df_trn_x['balance'], bins=[-1000, 0, 5000, 10000, 50000], labels=['negative', 'low', 'medium', 'high'])
df_tst_x['balance_group'] = pd.cut(df_tst_x['balance'], bins=[-1000, 0, 5000, 10000, 50000], labels=['negative', 'low', 'medium', 'high'])

# 3. 캠페인 참여 기간(duration * campaign)
df_trn_x['campaign_duration'] = df_trn_x['duration'] * df_trn_x['campaign']
df_tst_x['campaign_duration'] = df_tst_x['duration'] * df_tst_x['campaign']

In [56]:
for col in cat_cols:
    df_trn_x[col] = df_trn_x[col].astype('category')
    df_tst_x[col] = df_tst_x[col].astype('category')

In [57]:
from sklearn.preprocessing import LabelEncoder
import pandas as pd

# 범주형 변수+특성 엔지니어링한 변수를 Label Encoding으로 변환
cat_cols_with_new = cat_cols + ['age_group', 'balance_group'] 
label_encoders = {}  # 각 컬럼에 대한 LabelEncoder 객체 저장

for col in cat_cols_with_new:
    le = LabelEncoder()
    df_trn_x[col] = le.fit_transform(df_trn_x[col])  # 훈련 데이터 변환
    df_tst_x[col] = le.transform(df_tst_x[col])      # 테스트 데이터 변환
    label_encoders[col] = le

In [58]:
from sklearn.model_selection import StratifiedKFold

# StratifiedKFold로 훈련 데이터를 7개의 폴드로 나누기 (각 폴드마다 훈련과 검증 데이터가 나뉨)
skf = StratifiedKFold(n_splits=7, shuffle=True, random_state=42)

# 첫 번째 폴드로 분할 예시
for train_index, valid_index in skf.split(df_trn_x, df_trn_y):
    df_trn_x_fold, df_valid_x_fold = df_trn_x.iloc[train_index], df_trn_x.iloc[valid_index]
    df_trn_y_fold, df_valid_y_fold = df_trn_y.iloc[train_index], df_trn_y.iloc[valid_index]
    # 여기에서 df_trn_x_fold와 df_valid_x_fold를 사용하여 모델을 훈련하고 검증합니다.
    break  # 첫 번째 폴드만 예시로 보여주기 위해 반복을 한 번만 실행

In [60]:
model = lgbm.LGBMClassifier(verbose=-1)

In [61]:
n_positive = sum(df_trn_y_fold.iloc[:, 0] == 1)  # 양성 클래스의 샘플 수
n_negative = sum(df_trn_y_fold.iloc[:, 0] == 0)
# scale_pos_weight 설정 (양성/음성 비율)
scale_pos_weight = n_negative / n_positive

In [62]:
param_grid = {
    'iterations': np.arange(100, 2000, 100), 
    'learning_rate': np.arange(0.01,1.0,0.05),
    'l2_leaf_reg': [1, 3, 5, 10],
    'depth': np.arange(3,10,1),
    'subsample': np.arange(0.8, 1.0, 0.1),
    'scale_pos_weight' : [1,scale_pos_weight],
    'min_child_samples': [20, 50, 100]
}


In [63]:
from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV
skf = StratifiedKFold(n_splits=7, shuffle=True, random_state=42)

In [64]:
%%time
random_search = RandomizedSearchCV(estimator=model,
                                   param_distributions=param_grid, 
                                   scoring='f1', 
                                   cv=skf,
                                   refit=True)

CPU times: total: 0 ns
,Wall time: 997 µs


In [65]:

random_search.fit(df_trn_x_fold, df_trn_y_fold.values.ravel())

RandomizedSearchCV(cv=StratifiedKFold(n_splits=7, random_state=42, shuffle=True),
                   estimator=LGBMClassifier(verbose=-1),
                   param_distributions={'depth': array([3, 4, 5, 6, 7, 8, 9]),
                                        'iterations': array([ 100,  200,  300,  400,  500,  600,  700,  800,  900, 1000, 1100,
       1200, 1300, 1400, 1500, 1600, 1700, 1800, 1900]),
                                        'l2_leaf_reg': [1, 3, 5, 10],
                                        'learning_rate': array([0.01, 0.06, 0.11, 0.16, 0.21, 0.26, 0.31, 0.36, 0.41, 0.46, 0.51,
       0.56, 0.61, 0.66, 0.71, 0.76, 0.81, 0.86, 0.91, 0.96]),
                                        'min_child_samples': [20, 50, 100],
                                        'scale_pos_weight': [1,
                                                             0.999958251576003],
                                        'subsample': array([0.8, 0.9])},
                   scoring='f1')

In [66]:
best_model = random_search.best_estimator_

In [67]:
bagging_model = BaggingClassifier(estimator=best_model,
                                 n_estimators=50,  # 모델 개수
                                 random_state=42)

In [68]:
catboost_model = CatBoostClassifier(iterations=1000, 
                                    depth=10, 
                                    learning_rate=0.1, 
                                    scale_pos_weight=scale_pos_weight,
                                    loss_function='Logloss',
                                    verbose=False)

In [69]:
# 최적의 모델 학습 (cv에서 나온 성능을 기준으로 하이퍼파라미터 조정)
best_model2 =catboost_model.fit(df_trn_x_fold, df_trn_y_fold,
                               eval_set=(df_valid_x_fold,df_valid_y_fold),
                                early_stopping_rounds=50)

In [70]:
bagging_catboost_model = BaggingClassifier(estimator=best_model2,
                                          n_estimators=50, 
                                          random_state=42,
                                          n_jobs=-1)

In [71]:
from sklearn.ensemble import StackingClassifier
from sklearn.linear_model import LogisticRegression

stacking_model = StackingClassifier(
    estimators=[
        ('lightgbm_bagging', bagging_model),
        ('catboost_bagging', bagging_catboost_model)
    ],
    final_estimator=LogisticRegression()  # 메타 모델로 로지스틱 회귀 사용
)

In [72]:
stacking_model.fit(df_trn_x_fold, df_trn_y_fold.values.ravel())

StackingClassifier(estimators=[('lightgbm_bagging',
                                BaggingClassifier(estimator=LGBMClassifier(depth=4,
                                                                           iterations=1700,
                                                                           l2_leaf_reg=1,
                                                                           learning_rate=0.31000000000000005,
                                                                           min_child_samples=50,
                                                                           scale_pos_weight=0.999958251576003,
                                                                           subsample=0.9,
                                                                           verbose=-1),
                                                  n_estimators=50,
                                                  random_state=42)),
                               ('catboost_bagging',
                                BaggingClassifier(estimator=<catboost.core.CatBoostClassifier object at 0x0000018984765B10>,
                                                  n_estimators=50, n_jobs=-1,
                                                  random_state=42))],
                   final_estimator=LogisticRegression())

In [74]:
y_pred = stacking_model.predict(df_tst_x)

In [75]:
df_submission = pd.read_csv('submission_example.csv')

In [77]:
df_submission['label'] = y_pred

In [78]:
df_submission

,ID,label
0,test00001,0
1,test00002,0
2,test00003,0
3,test00004,1
4,test00005,0
...,...,...
13559,test13560,1
13560,test13561,0
13561,test13562,0
13562,test13563,0


In [79]:
(df_submission['label']==1).sum()

1857

In [80]:
filename = 'stacking smote early.csv'
df_submission.to_csv(filename, index=False)